# FastText - Turkce Metin Siniflandirma Kiyaslamasi

**Katman:** Klasik taban cizgisi

Alt-kelime gomme - Turkce gibi eklemeli dillerde avantajli (PeerJ makalesi bunu kullanmis).

Gorev: Turkce deprem ihbar metni -> 5 ust kategori (+ opsiyonel 1-5 aciliyet).  
Veri: `data/` (bkz `data/README.md`).  
Butun defterler `common.py` uzerinden AYNI bolme ve AYNI metrikleri kullanir.  
Metrik: Dogruluk, makro-F1, sinif bazli P/R, konfuzyon matrisi, **yanlis negatif orani**.

In [ ]:
# === COLAB KURULUM ===  (yerelde calistirinca atlanir)
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('/content/QuakeMind'):
    !git clone --depth 1 https://github.com/UtBird/QuakeMind.git /content/QuakeMind
if IN_COLAB:
    %cd /content/QuakeMind/model_benchmarks/04_turkce_metin
    %pip install -q fasttext-wheel
print('cwd:', os.getcwd())

## 1. Veri

In [ ]:
import sys, warnings; warnings.filterwarnings('ignore')
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import numpy as np, pandas as pd
import common as C

train_df, val_df, test_df = C.load_text_dataset()
labels, l2i, i2l = C.label_maps(train_df)
print('siniflar:', labels)
print('train/val/test:', len(train_df), len(val_df), len(test_df))
y_test = test_df['label'].map(l2i).values

## 2. FastText formatina yaz

In [ ]:
MODEL_NAME = "fasttext"
PARAMS = dict(epoch=25, lr=0.5, wordNgrams=2, dim=100)
import fasttext
def dump(df, path):
    with open(path, 'w') as f:
        for _, r in df.iterrows():
            f.write(f"__label__{r['label']} {r['text'].replace(chr(10),' ')}\n")
dump(train_df, 'ft_train.txt'); dump(test_df, 'ft_test.txt')

## 3. Egit ve tahmin et

In [ ]:
ft = fasttext.train_supervised('ft_train.txt', **PARAMS)
y_pred = np.array([l2i[ft.predict(t.replace(chr(10),' '))[0][0].replace('__label__','')]
                   for t in test_df['text']])

## 4. Sonuc

In [ ]:
m = C.classification_metrics(y_test, y_pred, labels)
print('accuracy :', round(m['accuracy'], 4))
print('macro-F1 :', round(m['macro_f1'], 4))
print('macro yanlis-negatif orani:', round(m['macro_fnr'], 4))
print()
print(m['report'])
lb = C.save_result(MODEL_NAME, m, extra={'params': PARAMS})
lb

In [ ]:
import matplotlib.pyplot as plt
cm = np.array(m['confusion_matrix'])
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
for (i, j), v in np.ndenumerate(cm): ax.text(j, i, int(v), ha='center', va='center')
ax.set_xlabel('tahmin'); ax.set_ylabel('gercek'); ax.set_title(f'{MODEL_NAME} konfuzyon')
plt.tight_layout(); plt.savefig(f'results/{MODEL_NAME}_cm.png', dpi=140); plt.show()